In [2]:
import os
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using", device, "device")

class Network(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten=nn.Flatten()
    self.sequence=nn.Sequential(nn.Linear(28*28,200), nn.ReLU(), nn.Linear(200,50), nn.ReLU(), nn.Linear(50,20), nn.ReLU(), nn.Linear(20,10))
  def forward(self, x):
    x=self.flatten(x)
    expected=self.sequence(x)
    return expected

model=Network().to(device)

input_size = 784
num_epochs = 2
batch_size = 100
learning_rate = 0.01

train_dataset = torchvision.datasets.MNIST(root='./data',
                                            train=True,
                                       transform=transforms.ToTensor(),
                                           download=True)
test_dataset = torchvision.datasets.MNIST(root='./data',
                                           train=False,
                                           transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                            batch_size=batch_size,
                                            shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                           batch_size=batch_size,
                                           shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

n_total_steps = len(train_loader)
for epoch in range(num_epochs):
     for i, (images, labels) in enumerate(train_loader):
         images = images.reshape(-1, 28*28).to(device)
         labels = labels.to(device)
         outputs = model(images)
         loss = criterion(outputs, labels)
         optimizer.zero_grad()
         loss.backward()
         optimizer.step()
         if (i+1) % 100 == 0:
             print (f'Epoch [{epoch+1}/{num_epochs}], Step[{i+1}/{n_total_steps}], Loss: {loss.item():.4f}')

with torch.no_grad():
     n_correct = 0
     n_samples = 0
     for images, labels in test_loader:
         images = images.reshape(-1, 28*28).to(device)
         labels = labels.to(device)
         outputs = model(images)
         _, predicted = torch.max(outputs.data, 1)
         n_samples += labels.size(0)
         n_correct += (predicted == labels).sum().item()

acc = 100.0 * n_correct / n_samples
print(f'Accuracy of the network on the 10000 test images: {acc} %')

Using cpu device
Epoch [1/5], Step[100/600], Loss: 0.3581
Epoch [1/5], Step[200/600], Loss: 0.2651
Epoch [1/5], Step[300/600], Loss: 0.2589
Epoch [1/5], Step[400/600], Loss: 0.3084
Epoch [1/5], Step[500/600], Loss: 0.1583
Epoch [1/5], Step[600/600], Loss: 0.2738
Epoch [2/5], Step[100/600], Loss: 0.0384
Epoch [2/5], Step[200/600], Loss: 0.0658
Epoch [2/5], Step[300/600], Loss: 0.0667
Epoch [2/5], Step[400/600], Loss: 0.1307
Epoch [2/5], Step[500/600], Loss: 0.1223
Epoch [2/5], Step[600/600], Loss: 0.0611
Epoch [3/5], Step[100/600], Loss: 0.1043
Epoch [3/5], Step[200/600], Loss: 0.2188
Epoch [3/5], Step[300/600], Loss: 0.0266
Epoch [3/5], Step[400/600], Loss: 0.1726
Epoch [3/5], Step[500/600], Loss: 0.1142
Epoch [3/5], Step[600/600], Loss: 0.0920
Epoch [4/5], Step[100/600], Loss: 0.1643
Epoch [4/5], Step[200/600], Loss: 0.0134
Epoch [4/5], Step[300/600], Loss: 0.1065
Epoch [4/5], Step[400/600], Loss: 0.1422
Epoch [4/5], Step[500/600], Loss: 0.0875
Epoch [4/5], Step[600/600], Loss: 0.1226